# fedOPD — build the canonical table

Transformation only. Reads the 96 validated monthly RTT extracts and produces one tidy table at **provider × department × month**, plus the rectangular panel the federation trains on.

Run after `inspect_dataset.ipynb`, which establishes the facts relied on here.

## What this builds

**19 series per provider:** the 18 named specialties that span all 96 months, plus `X_OTHER`, a merged residual worth about a quarter of demand.

**All providers retained**, trust and independent alike, with a `sector` column. Sector filtering is a downstream one-line panel decision, never baked in here.

## The two era reconciliations

NHS England changed the return in April 2021 in two independent ways, both the same shape — coarse before, finer after. Both reconcile to the **coarser** form, because the early era cannot be disaggregated.

| | pre-2021 | post-2021 | reconciled to |
|---|---|---|---|
| Long waits | single `Gt 52 Weeks` column | weekly bands 52→104 plus `Gt 104` | one `>52` figure |
| Residual specialties | single `X01 Other` | five buckets `X02`–`X06` | one `X_OTHER` series |

Columns are chosen **by name with an explicit era branch**, never by position.

## Standing asserts

Every trap found during inspection fails loudly here rather than silently:

1. **Full-coverage identity** — 18 named + `X_OTHER` = `C_999`, per provider, part type and month.
2. **Boundary continuity at 2021-04** — measured against *local* variation, not global.
3. **Code-only joins** — treatment functions keyed on code; names changed at the break, codes did not.
4. **Key uniqueness** — one row per provider × department × month.

In [ ]:
import re                                         # parse the month and band edges out of names
from pathlib import Path                          # filesystem paths
from datetime import datetime                     # sortable month keys

import pandas as pd                               # all tabular work
import matplotlib.pyplot as plt                   # the continuity plots

DATA = Path("../data")
RTT_CSV = DATA / "rtt_csv"                        # the 96 validated monthly files
REF = DATA / "reference"                          # census + lookups from inspect_dataset
OUT = DATA / "canonical"                          # everything this notebook produces
OUT.mkdir(parents=True, exist_ok=True)            # no-op if it already exists

AGG = "C_999"                                     # published aggregate row — never a department
OTHER = "X_OTHER"                                 # our merged residual series
ERA = datetime(2021, 4, 1)                        # first month of the post-change schema

MONTHS = "JANUARY FEBRUARY MARCH APRIL MAY JUNE JULY AUGUST SEPTEMBER OCTOBER NOVEMBER DECEMBER".split()

def month_of(name):
    """'RTT-OCTOBER-2019-full-extract.csv' -> datetime(2019, 10, 1)."""
    m = re.search(r"(" + "|".join(MONTHS) + r")-(\d{4})", name.upper())   # month word + year
    return datetime(int(m.group(2)), MONTHS.index(m.group(1)) + 1, 1)     # index+1 = month number

csvs = sorted(RTT_CSV.glob("*.csv"), key=lambda p: month_of(p.name))      # chronological order
census = pd.read_csv(REF / "provider_census.csv")                        # built by inspect_dataset
SECTOR = dict(zip(census.code, census.sector))                           # code -> public/independent
ICB = dict(zip(census.code, census.icb_parent))                          # code -> ICB where one exists
PANEL = set(census[census.panel_member].code)                            # the 126 balanced-panel trusts
print(f"{len(csvs)} monthly files | {len(census)} providers | {len(PANEL)} panel trusts")

## Helpers

Four small functions carry every era-dependent decision, so the branch lives in one visible place instead of being scattered through the build.

In [ ]:
def week_of(col):
    """Lower edge of a band column. 'Gt 18 To 19 Weeks SUM 1' -> 18, 'Gt 104 Weeks SUM 1' -> 104."""
    return int(re.match(r"Gt (\d+)", col).group(1))           # the first number in the name

def bands_from(cols, lo):
    """Band columns at or above `lo` weeks, selected BY NAME so both eras work unchanged:
       pre-2021,  lo=52 -> ['Gt 52 Weeks SUM 1']           the single open-ended bucket
       post-2021, lo=52 -> ['Gt 52 To 53', ..., 'Gt 104']  weekly bands plus the tail
       The 18-52 bands are identical in both eras, so lo=18 needs no branch at all."""
    return [c for c in cols if "Weeks SUM" in c               # waiting-time band columns only
            and week_of(c) >= lo]                             # at or above the threshold

def dept_of(code):
    """Treatment-function code -> series name. X01..X06 all collapse into one residual."""
    return OTHER if str(code).startswith("X") else code       # named specialties pass through

def header_of(path):
    """Column names of a csv, without reading the body."""
    return open(path, newline="").readline().rstrip("\r\n").replace('"', "").split(",")

# print what the rule selects in each era, so the branch is visible rather than implied
for label, f in [("pre-2021", csvs[0]), ("post-2021", csvs[-1])]:
    cols = header_of(f)                                       # this month's schema
    g52 = bands_from(cols, 52)                                # the era-dependent set
    print(f"{label:10s} >18 uses {len(bands_from(cols, 18)):3d} cols | "
          f">52 uses {len(g52):3d} cols -> {g52[0]}{' ... ' + g52[-1] if len(g52) > 1 else ''}")

## The build pass

One read of all 96 files. Band columns are read **only for `Part_2`**, the waiting list — the only part type whose long-wait detail is used. Reading bands for the other four would multiply the IO for columns we then discard.

This is the slow cell.

In [ ]:
KEYS = ["Provider Org Code", "Provider Org Name", "Provider Parent Org Code",
        "RTT Part Type", "Treatment Function Code", "Total All"]      # always needed

totals, longwaits, coverage = [], [], []          # three small frames per month, concatenated after
for p in csvs:
    mth = month_of(p.name)                        # this file's month
    cols = header_of(p)                           # this file's schema, 69 or 121 columns
    g18 = bands_from(cols, 18)                    # >18w columns, era-stable
    g52 = bands_from(cols, 52)                    # >52w columns, era-dependent
    d = pd.read_csv(p, usecols=lambda c: c in set(KEYS) | set(g18), low_memory=False)

    # --- coverage input: set the published C_999 rows aside, then drop them from the data ---
    is_agg = d["Treatment Function Code"].eq(AGG)                      # the aggregate rows
    published = (d[is_agg].groupby(["Provider Org Code", "RTT Part Type"])["Total All"]
                          .sum().rename("published"))                  # what NHS says the total is
    d = d[~is_agg].copy()                                              # real specialties only
    d["dept"] = d["Treatment Function Code"].map(dept_of)              # X01..X06 -> X_OTHER

    ours = (d.groupby(["Provider Org Code", "RTT Part Type"])["Total All"]
             .sum().rename("ours"))                                    # what our 19 series sum to
    chk = pd.concat([published, ours], axis=1).fillna(0).reset_index()
    chk["month"] = mth
    coverage.append(chk)                                               # checked by Assert 1

    # --- headline totals: one value per provider x dept x part type ---
    t = (d.groupby(["Provider Org Code", "Provider Org Name",
                    "Provider Parent Org Code", "dept", "RTT Part Type"])["Total All"]
          .sum().rename("total").reset_index())
    t["month"] = mth
    totals.append(t)

    # --- long waits: Part_2 only, because only the waiting list needs band detail ---
    w = d[d["RTT Part Type"] == "Part_2"]                              # incomplete pathways
    lw = pd.DataFrame({
        "Provider Org Code": w["Provider Org Code"],
        "dept": w["dept"],
        "waiting_gt18": w[g18].fillna(0).sum(axis=1),                  # sum the >=18w bands
        "waiting_gt52": w[g52].fillna(0).sum(axis=1),                  # era-reconciled >=52w
    })
    lw = lw.groupby(["Provider Org Code", "dept"]).sum().reset_index() # collapse to the key
    lw["month"] = mth
    longwaits.append(lw)

totals = pd.concat(totals, ignore_index=True)                          # one row per part type
longwaits = pd.concat(longwaits, ignore_index=True)
coverage = pd.concat(coverage, ignore_index=True)
print(f"{len(totals):,} provider x dept x part x month rows | {len(longwaits):,} long-wait rows")

## Assert 1 — full-coverage identity

18 named + `X_OTHER` must equal the published `C_999`, for every provider, part type and month. Stronger than the roster-drift check in `inspect_dataset.ipynb`: it proves the 19 series account for **all** reported activity, so the hierarchy reconciles exactly rather than approximately.

In [ ]:
coverage["diff"] = coverage["ours"] - coverage["published"]    # zero everywhere if coverage is total
bad = coverage[coverage["diff"].abs() > 0.5]                   # tolerance for float noise only
print(f"groups compared : {len(coverage):,}")
print(f"mismatched      : {len(bad)}")
if len(bad):
    print(bad.head(15).to_string(index=False))                 # show exactly what broke
assert len(bad) == 0, "the 19 series do not reproduce C_999 — coverage is incomplete"
print("\nASSERT 1 PASSED: 18 named + X_OTHER = C_999 everywhere. Coverage is total.")

## Assemble the canonical table

Pivot the five part types into columns, attach the long-wait detail, then the census flags.

In [ ]:
IDX = ["month", "Provider Org Code", "Provider Org Name", "Provider Parent Org Code", "dept"]

# one column per RTT part type; missing combinations become 0 rather than NaN
wide = totals.pivot_table(index=IDX, columns="RTT Part Type", values="total",
                          fill_value=0).reset_index()
wide.columns.name = None                                       # drop the pivot's column-axis label

canon = wide.rename(columns={
    "Provider Org Code": "provider_code",
    "Provider Org Name": "provider_name",
    "Provider Parent Org Code": "icb_reported",                 # parent as stated in the RTT file
    "Part_3": "demand",                                         # new RTT periods = demand signal
    "Part_1A": "completed_admitted",                            # completed pathways, admitted
    "Part_1B": "completed_nonadmitted",                         # completed pathways, non-admitted
    "Part_2": "waiting",                                        # incomplete pathways = waiting list
    "Part_2A": "waiting_dta",                                   # of those, with a decision to admit
})

# attach >18w / >52w detail; zero where a provider/dept had no waiting list that month
canon = canon.merge(longwaits.rename(columns={"Provider Org Code": "provider_code"}),
                    on=["month", "provider_code", "dept"], how="left")
canon[["waiting_gt18", "waiting_gt52"]] = canon[["waiting_gt18", "waiting_gt52"]].fillna(0)

canon["completed"] = canon.completed_admitted + canon.completed_nonadmitted   # total throughput
canon["sector"] = canon.provider_code.map(SECTOR)               # public / independent
canon["icb_ods"] = canon.provider_code.map(ICB)                 # ODS confirmation of the parent
canon["panel_member"] = canon.provider_code.isin(PANEL)         # flag, never a filter
canon["in_roster"] = canon.dept.ne(OTHER)                       # False only for X_OTHER
canon = canon.sort_values(["provider_code", "dept", "month"]).reset_index(drop=True)

# Assert 4 — the key must be unique
dup = int(canon.duplicated(["provider_code", "dept", "month"]).sum())
assert dup == 0, f"{dup} duplicate provider/dept/month rows"
print(f"canonical rows : {len(canon):,}")
print(f"providers      : {canon.provider_code.nunique()}")
print(f"departments    : {canon.dept.nunique()}  (18 named + {OTHER})")
print(f"months         : {canon.month.nunique()}")
canon.head()

## Assert 2 — boundary continuity at 2021-04

Both reconciliations must be invisible in the joined series. The comparison uses **local** variation — the months either side of the break — not a global dispersion statistic. The `>52` series sits near zero through the whole pre-covid period, so a global median is dominated by quiet months and would make any step at a turbulent join look like a discontinuity when it is not.

In [ ]:
def boundary_ratio(series, window=6):
    """Step at the era join, expressed in units of LOCAL month-to-month variation."""
    s = series.sort_index()
    prev = pd.Timestamp(ERA) - pd.offsets.MonthBegin(1)        # last pre-change month
    step = s.loc[ERA] - s.loc[prev]                            # the join step being judged
    diffs = s.diff().abs()                                     # every month-to-month change
    lo = pd.Timestamp(ERA) - pd.offsets.MonthBegin(window)     # window months before
    hi = pd.Timestamp(ERA) + pd.offsets.MonthBegin(window)     # window months after
    local = diffs.loc[lo:hi].drop(index=ERA).median()          # exclude the step from its own yardstick
    return step, local, abs(step) / local

other_demand = canon[canon.dept == OTHER].groupby("month").demand.sum()    # the merged residual
gt52_waiting = canon.groupby("month").waiting_gt52.sum()                   # reconciled long waits

for label, s in [("X_OTHER demand", other_demand), (">52 waiting", gt52_waiting)]:
    step, local, ratio = boundary_ratio(s)
    print(f"{label:16s} step {step:>12,.0f}   local |change| {local:>12,.0f}   ratio {ratio:.2f}x")
    assert ratio < 3, f"{label} steps at the era boundary — the reconstruction is wrong"
print("\nASSERT 2 PASSED: neither reconstruction introduces a step at 2021-04.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(other_demand.index, other_demand / 1e3, lw=1.3)
ax[0].axvline(ERA, color="grey", ls=":")                       # the recode boundary
ax[0].set_title("X_OTHER new referrals (X01 -> X02..X06 merged)")
ax[0].set_ylabel("thousands")
ax[1].plot(gt52_waiting.index, gt52_waiting / 1e3, lw=1.3, color="firebrick")
ax[1].axvline(ERA, color="grey", ls=":")                       # same boundary, other reconciliation
ax[1].set_title(">52 week waits (era-reconciled)")
ax[1].set_ylabel("thousands")
plt.tight_layout()

## The panel cube

The federation trains on a rectangular 126 × 19 × 96 cube. Cells absent from the source are filled with zero, but **most of that fill is structural** — a trust that does not provide neurosurgery has no neurosurgery rows, ever. Those series are counted and flagged here so phase 3 can exclude them in one line instead of training on constant zeros.

In [ ]:
MEASURES = ["demand", "completed_admitted", "completed_nonadmitted", "completed",
            "waiting", "waiting_gt18", "waiting_gt52", "waiting_dta"]

# every (trust, dept, month) combination a balanced panel should contain
full = pd.MultiIndex.from_product(
    [sorted(PANEL), sorted(canon.dept.unique()), sorted(canon.month.unique())],
    names=["provider_code", "dept", "month"])

panel = (canon[canon.panel_member]                             # panel trusts only
         .set_index(["provider_code", "dept", "month"])
         .reindex(full))                                       # absent combinations become NaN rows
filled = int(panel.demand.isna().sum())                        # how many cells were absent
panel[MEASURES] = panel[MEASURES].fillna(0)                    # absent = no activity = 0
panel = panel.reset_index()
panel["sector"] = "public"                                     # public by construction
panel["in_roster"] = panel.dept.ne(OTHER)                      # False only for X_OTHER

print(f"panel cube : {len(panel):,} rows = {len(PANEL)} trusts x "
      f"{panel.dept.nunique()} depts x {panel.month.nunique()} months")
print(f"cells filled with zero : {filled:,} ({filled / len(panel) * 100:.2f}% of the cube)")

In [ ]:
# how much of that fill is a service that never existed, vs a month that went unreported?
lifetime = panel.groupby(["provider_code", "dept"]).demand.sum()    # total demand per series
never = lifetime.eq(0)                                             # never any activity at all
print(f"trust x dept series in the cube  : {len(lifetime):,}")
print(f"  never any demand (structural)  : {int(never.sum()):,} ({never.mean() * 100:.1f}%)")
print(f"  genuinely active series        : {int((~never).sum()):,}")

# for the active ones, how complete is the history?
active_months = panel[panel.demand > 0].groupby(["provider_code", "dept"]).size()
print(f"  active series: median {active_months.median():.0f}/96 months with demand, "
      f"min {active_months.min()}")

# flag it in the cube so phase 3 filters in one line instead of rediscovering this
keys = list(zip(panel.provider_code, panel.dept))                  # the series each row belongs to
exists = (~never).to_dict()                                        # (provider, dept) -> bool
panel["service_exists"] = [exists[k] for k in keys]
print(f"\nrows flagged service_exists=False: {int((~panel.service_exists).sum()):,}")

## The backlog accounting identity

The README's decision layer assumes `backlog ≈ previous backlog + demand − completions`. That is an approximation: pathways also leave the list without a completion — removals, transfers, validation exercises, deaths, patients who decline. Measuring the residual tells the agent designer the size of the term they would be ignoring.

In [ ]:
nat = panel.groupby("month")[["demand", "completed", "waiting"]].sum().sort_index()
nat["predicted"] = nat.waiting.shift(1) + nat.demand - nat.completed    # the stated identity
nat["residual"] = nat.waiting - nat.predicted                           # what the identity misses
nat["residual_pct_list"] = nat.residual / nat.waiting * 100             # as % of the waiting list
nat["residual_pct_demand"] = nat.residual / nat.demand * 100            # as % of monthly demand

r = nat.residual.dropna()                                              # first month has no previous
print(f"months measured        : {len(r)}")
print(f"mean residual          : {r.mean():,.0f} per month")
print(f"  as % of waiting list : {nat.residual_pct_list.mean():.2f}%")
print(f"  as % of demand       : {nat.residual_pct_demand.mean():.2f}%")
print(f"negative in            : {int((r < 0).sum())} of {len(r)} months")   # one-directional?
print()
print("A persistent one-sided residual is a missing term, not noise: pathways leave the")
print("list without a recorded completion, so the identity over-predicts the backlog")
print("every month and the error compounds across a multi-month projection horizon.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(nat.index, nat.waiting / 1e6, lw=1.4, label="actual")
ax[0].plot(nat.index, nat.predicted / 1e6, lw=1.2, ls="--", label="identity prediction")
ax[0].set_title("Waiting list: actual vs accounting identity")
ax[0].set_ylabel("millions"); ax[0].legend()
ax[1].plot(nat.index, nat.residual_pct_list, lw=1.3, color="firebrick")
ax[1].axhline(0, color="grey", lw=0.8)                         # zero = the identity holds exactly
ax[1].set_title("Identity residual as % of waiting list")
ax[1].set_ylabel("%")
plt.tight_layout()

## Write the outputs

In [ ]:
canon.to_parquet(OUT / "rtt_canonical.parquet", index=False)   # all providers, flags not filters
panel.to_parquet(OUT / "rtt_panel.parquet", index=False)       # rectangular 126 x 19 x 96 cube
nat.to_csv(OUT / "national_monthly.csv")                       # national series + identity check

for f in sorted(OUT.glob("*")):
    print(f"{f.name:26s} {f.stat().st_size / 1e6:7.1f} MB")
print()
print("canonical schema:")
print(canon.dtypes.to_string())